# 00 · Setup
One-time workspace setup (module 04): schemas, volumes, shared DQ tables, and moving uploaded files into one folder per table.

**Before running:** in Catalog Explorer, upload the 9 CSVs and `DQ_ANSWER_KEY.json` into
`/Volumes/<catalog>/landing/hcm_raw/_upload/` (run cell 2 first so the volume exists). Re-running this notebook is safe.

In [0]:
from hcm_utils import load_config, resolve_paths

dbutils.widgets.text("catalog", "workspace")
catalog = dbutils.widgets.get("catalog")
cfg = load_config()
paths = resolve_paths(cfg, catalog)

In [0]:
for stmt in [
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.landing COMMENT 'Raw files from Oracle HCM extracts'",
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze  COMMENT 'Raw, append-only, all STRING + audit columns'",
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver  COMMENT 'Cleansed, conformed, SCD2, DQ-logged'",
    f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold    COMMENT 'Star schema for BI and Genie'",
    f"CREATE VOLUME IF NOT EXISTS {catalog}.landing.hcm_raw     COMMENT 'One sub-folder per source table'",
    f"CREATE VOLUME IF NOT EXISTS {catalog}.landing.reference   COMMENT 'DQ answer key and other reference files'",
    f"CREATE VOLUME IF NOT EXISTS {catalog}.landing.checkpoints COMMENT 'Auto Loader schema + checkpoint state'",
]:
    spark.sql(stmt)
dbutils.fs.mkdirs(paths["upload"])

In [0]:
# Created up front so the 5 parallel Silver tasks only ever append (concurrent first-time CREATEs conflict).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.dq_issue_log (
  SourceFile STRING, RowRef STRING, IssueType STRING, Description STRING, ActionTaken STRING,
  RunId STRING, DetectedAt TIMESTAMP)
COMMENT 'Every detected data-quality defect, one row per offending source row and rule'""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.quarantine (
  SourceFile STRING, RowRef STRING, Reason STRING, RawRecord STRING,
  RunId STRING, QuarantinedAt TIMESTAMP)
COMMENT 'Rows rejected by Silver, with the raw record as JSON'""")

In [0]:
# Move uploaded files into the layout Auto Loader expects: hcm_raw/<table>/<FILE>.csv
uploaded = {f.name: f.path for f in dbutils.fs.ls(paths["upload"])}
for t in cfg["bronze_tables"]:
    name = f"{t.upper()}.csv"
    if name in uploaded:
        dbutils.fs.mv(uploaded[name], f"{paths['landing']}/{t}/{name}")
        print(f"moved {name} -> {t}/")
if "DQ_ANSWER_KEY.json" in uploaded:
    dbutils.fs.mv(uploaded["DQ_ANSWER_KEY.json"], paths["dq_answer_key"])
    print("moved DQ_ANSWER_KEY.json -> reference volume")

In [0]:
def exists(path: str) -> bool:
    try:
        return len(dbutils.fs.ls(path)) > 0
    except Exception:
        return False

missing = [t for t in cfg["bronze_tables"] if not exists(f"{paths['landing']}/{t}/{t.upper()}.csv")]
if not exists(paths["dq_answer_key"]):
    missing.append("DQ_ANSWER_KEY.json")
assert not missing, f"Upload these to {paths['upload']} and re-run: {missing}"
print("Setup complete: 9 source folders + answer key in place.")